# 02 · Sanity Checks: Environment Verification & Reward Unit Tests

Before spending time on training we check two things:

- **Part A — Environment** (from `code/verify_env.py`): the CityLearn dataset loads, the wrapped environment has the
  expected observation / action shapes, the observation indices in the config point at the right features, and the
  cost / emission arrays and solar sign behave the way the rest of the code assumes.
- **Part B — Reward functions** (from `code/test_rewards.py`): unit tests on a mock environment with hand-computed
  expected values for all three reward formulas.

In [1]:
# Load all shared config, reward functions, environment wrapper and helpers from notebook 01
%run ./01_setup_and_core.ipynb

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


Configuration loaded. Project root: D:\DEV\Projects\1_ResumeWorthy\SmartGridRL


## Part A — Environment verification
### A1. Available datasets

In [2]:
from citylearn.data import DataSet

datasets = DataSet.get_names()
print(f"{len(datasets)} datasets available")
print(f"Target dataset '{DATASET_NAME}' available: {DATASET_NAME in datasets}")
assert DATASET_NAME in datasets

18 datasets available
Target dataset 'citylearn_challenge_2022_phase_1' available: True


### A2. Build the wrapped environment and check the spaces
Central agent → one observation vector for all buildings, one action per building battery.

In [3]:
env = make_env(reward_type='default', seed=42, verbose=True)
print(f"\nObservation space shape (centralised): {env.observation_space.shape}")
print(f"Action space shape (centralised):      {env.action_space.shape}")
print(f"Action range: {env.action_space.low} to {env.action_space.high}")
print(f"Episode length: {env.env.time_steps} hourly steps")

Wrapper initialized: reward_type=default, lambda=None
Carbon intensity bounds detected: min=0.0704, max=0.2818

Observation space shape (centralised): (60,)
Action space shape (centralised):      (5,)
Action range: [-0.78125 -0.78125 -0.78125 -0.78125 -0.78125] to [0.78125 0.78125 0.78125 0.78125 0.78125]
Episode length: 8760 hourly steps


### A3. Do the configured observation indices point at the right features?

In [4]:
obs_names = env.env.observation_names[0]
obs, _ = env.reset()

index_check = pd.DataFrame([
    {'config name': name, 'index': idx, 'CityLearn feature': obs_names[idx], 'value at step 0': obs[idx]}
    for name, idx in [
        ('CARBON_INTENSITY_IDX', CARBON_INTENSITY_IDX),
        ('NON_SHIFTABLE_LOAD_IDX', NON_SHIFTABLE_LOAD_IDX),
        ('SOLAR_GENERATION_IDX', SOLAR_GENERATION_IDX),
        ('SOC_IDX', SOC_IDX),
        ('NET_ELECTRICITY_CONSUMPTION_IDX', NET_ELECTRICITY_CONSUMPTION_IDX),
        ('PRICING_IDX', PRICING_IDX),
    ]
])
print(f"Observation vector has {len(obs_names)} features")
index_check

Observation vector has 60 features


,config name,index,CityLearn feature,value at step 0
0,CARBON_INTENSITY_IDX,19,carbon_intensity,0.170724
1,NON_SHIFTABLE_LOAD_IDX,20,non_shiftable_load,2.275800
2,SOLAR_GENERATION_IDX,21,solar_generation,0.000000
3,SOC_IDX,22,electrical_storage_soc,0.000000
4,NET_ELECTRICITY_CONSUMPTION_IDX,23,net_electricity_consumption,2.275800
5,PRICING_IDX,24,electricity_pricing,0.220000


### A4. Are cost / emission arrays read at the right time step?
The reward functions read `[-1]` of each building's cost / emission history right after `env.step()`.
We step with alternating charge (+0.5) / discharge (−0.5) actions: if `[-1]` is the step just taken, cost should
jump up on charging steps and down on discharging steps, and the per-step sum must equal the sum of the
history (excluding the placeholder at index 0).

In [5]:
env.reset(seed=42)
rows = []
for t in range(6):
    a = np.full(env.action_space.shape, 0.5 if t % 2 == 0 else -0.5, dtype=np.float32)
    _, _, _, _, info = env.step(a)
    rows.append({'step': t, 'action': a[0], 'step cost (all buildings)': info['step_metrics']['cost'],
                 'step carbon (all buildings)': info['step_metrics']['carbon']})
display(pd.DataFrame(rows))

hist_cost = sum(float(np.sum(b.net_electricity_consumption_cost[1:])) for b in env.env.buildings)
print(f"Sum of per-step costs: {sum(r['step cost (all buildings)'] for r in rows):.6f}")
print(f"Sum of cost history:   {hist_cost:.6f}")

,step,action,step cost (all buildings),step carbon (all buildings)
0,0,0.5,4.546494,3.251137
1,1,-0.5,-2.467944,0.000000
2,2,0.5,4.042625,2.773698
3,3,-0.5,-2.453604,0.000000
4,4,0.5,4.298301,3.088686
5,5,-0.5,-2.421422,0.000000


Sum of per-step costs: 5.544451
Sum of cost history:   5.544451


### A5. Sign of `solar_generation`
Self-consumption needs total solar generated. If CityLearn stores generation as negative numbers, summing it
gives a negative total — which is why the original evaluation reported 100% self-consumption for every method.

In [6]:
env.reset(seed=42)
for _ in range(24):  # one full day, battery idle
    env.step(np.zeros(env.action_space.shape, dtype=np.float32))
solar = np.concatenate([np.array(b.solar_generation[1:]) for b in env.env.buildings])
print(f"solar_generation over the first 24 h, all buildings: min = {solar.min():.3f}, max = {solar.max():.3f}")
print(f"Stored as {'NEGATIVE' if solar.min() < 0 else 'non-negative'} values -> self-consumption must use abs(solar_generation)")

solar_generation over the first 24 h, all buildings: min = -3.039, max = -0.000
Stored as NEGATIVE values -> self-consumption must use abs(solar_generation)


## Part B — Reward function unit tests
A mock environment exposes only what the reward functions read: per-building cost and emission histories,
the carbon-intensity series and `time_step`. Two buildings, current step values cost = 0.5 + 0.3 = **0.8**,
carbon = 0.4 + 0.2 = **0.6**.

In [7]:
import unittest

class MockBuilding:
    """Simulates a CityLearn building's cost and emission history."""
    def __init__(self, cost_history, emission_history):
        self.net_electricity_consumption_cost = cost_history
        self.net_electricity_consumption_emission = emission_history

class MockCarbonIntensity:
    """Simulates the carbon intensity time-series container."""
    def __init__(self, intensities):
        self._carbon_intensity = intensities

class MockEnvironment:
    """Simulates a CityLearnEnv instance; carbon intensity lives on building 0, as in CityLearn."""
    def __init__(self, cost_histories, emission_histories, intensities, time_step=1):
        self.buildings = [MockBuilding(c, e) for c, e in zip(cost_histories, emission_histories)]
        self.buildings[0].carbon_intensity = MockCarbonIntensity(intensities)
        self.time_step = time_step

### Test cases and expected values

| Test | Setting | Expected |
|---|---|---|
| Step metrics | sum of last elements | cost 0.8, carbon 0.6 |
| Carbon intensity | `time_step=1` → index 0; `time_step=4` → index 3 | 0.10; 0.25 |
| Cost-only | $-0.8$ | −0.8 |
| Static, λ = 2.5 | $-0.8 - 2.5 \times 0.6$ | −2.3 |
| Dynamic, λ₀ = 2, $I \in [0.10, 0.30]$ | max intensity → $\hat I = 1$ | −2.0 (= static) |
| | min intensity → $\hat I = 0$ | −0.8 (= cost-only) |
| | mid intensity 0.20 → $\hat I = 0.5$ | −1.4 |
| | 0.35 (above max) → clipped to 1 | −2.0 |
| | 0.05 (below min) → clipped to 0 | −0.8 |

In [8]:
class TestRewardShaping(unittest.TestCase):

    def setUp(self):
        self.cost_histories = [[1.2, 0.5], [0.8, 0.3]]      # current step: 0.5 + 0.3 = 0.8
        self.emission_histories = [[1.0, 0.4], [0.6, 0.2]]  # current step: 0.4 + 0.2 = 0.6
        self.intensities = [0.10, 0.15, 0.20, 0.25, 0.30, 0.10, 0.10, 0.10, 0.10, 0.10]

    def test_step_metrics_extraction(self):
        env = MockEnvironment(self.cost_histories, self.emission_histories, self.intensities)
        cost, carbon = get_step_metrics(env)
        self.assertEqual(cost, 0.8)
        self.assertAlmostEqual(carbon, 0.6)

    def test_carbon_intensity_extraction(self):
        env = MockEnvironment(self.cost_histories, self.emission_histories, self.intensities, time_step=1)
        self.assertEqual(get_carbon_intensity(env), 0.10)
        env2 = MockEnvironment(self.cost_histories, self.emission_histories, self.intensities, time_step=4)
        self.assertEqual(get_carbon_intensity(env2), 0.25)

    def test_cost_only_reward(self):
        env = MockEnvironment(self.cost_histories, self.emission_histories, self.intensities)
        self.assertEqual(cost_only_reward(env, None), -0.8)

    def test_static_carbon_reward(self):
        env = MockEnvironment(self.cost_histories, self.emission_histories, self.intensities)
        self.assertAlmostEqual(static_carbon_reward(env, None, 2.5), -2.3)

    def test_dynamic_carbon_reward_normalization_and_reduction(self):
        carbon_min, carbon_max, lambda_base = 0.10, 0.30, 2.0

        # A: max intensity -> reduces to static reward
        env_max = MockEnvironment(self.cost_histories, self.emission_histories, self.intensities, time_step=5)
        r_max = dynamic_carbon_reward(env_max, None, lambda_base, carbon_min, carbon_max)
        self.assertAlmostEqual(r_max, -2.0)
        self.assertEqual(r_max, static_carbon_reward(env_max, None, lambda_base))

        # B: min intensity -> reduces to cost-only reward
        env_min = MockEnvironment(self.cost_histories, self.emission_histories, self.intensities, time_step=1)
        self.assertAlmostEqual(dynamic_carbon_reward(env_min, None, lambda_base, carbon_min, carbon_max), -0.8)

        # C: midpoint intensity -> half the carbon weight
        env_mid = MockEnvironment(self.cost_histories, self.emission_histories, self.intensities, time_step=3)
        self.assertAlmostEqual(dynamic_carbon_reward(env_mid, None, lambda_base, carbon_min, carbon_max), -1.4)

        # D: above max -> clipped to 1
        env_high = MockEnvironment(self.cost_histories, self.emission_histories, [0.35], time_step=1)
        self.assertAlmostEqual(dynamic_carbon_reward(env_high, None, lambda_base, carbon_min, carbon_max), -2.0)

        # E: below min -> clipped to 0
        env_low = MockEnvironment(self.cost_histories, self.emission_histories, [0.05], time_step=1)
        self.assertAlmostEqual(dynamic_carbon_reward(env_low, None, lambda_base, carbon_min, carbon_max), -0.8)

### Run the tests

In [9]:
test_result = unittest.main(argv=['ignored'], exit=False, verbosity=2).result
assert test_result.wasSuccessful(), "Reward unit tests failed"

test_carbon_intensity_extraction (__main__.TestRewardShaping) ... 

ok
test_cost_only_reward (__main__.TestRewardShaping) ... 

ok
test_dynamic_carbon_reward_normalization_and_reduction (__main__.TestRewardShaping) ... 

ok
test_static_carbon_reward (__main__.TestRewardShaping) ... 

ok
test_step_metrics_extraction (__main__.TestRewardShaping) ... 

ok

----------------------------------------------------------------------
Ran 5 tests in 0.003s

OK
